# 21: Random Forest Baseline Model

Trains a Random Forest Regressor baseline for daily COVID-19 case predictions across NJ ZIP codes with out-of-sample temporal validation and target leakage protection.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

print("✅ Core libraries imported successfully!")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def calc_rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2)))

def calc_mae(y_true, y_pred):
    return float(np.mean(np.abs(np.array(y_true) - np.array(y_pred))))

def find_data_file(filename):
    possible_paths = [
        filename,
        os.path.join(".", filename),
        os.path.join("..", filename),
        os.path.join("Weikun", filename),
        os.path.join("..", "Weikun", filename),
        os.path.join("Ayaan", filename),
        os.path.join("..", "Ayaan", filename),
        os.path.join("nj_daily_feature_outputs", filename),
        os.path.join("Weikun", "nj_daily_feature_outputs", filename),
        os.path.join("njny_commute_outputs", filename),
        os.path.join("Weikun", "njny_commute_outputs", filename)
    ]
    for p in possible_paths:
        if os.path.exists(p):
            return p
    for root, dirs, files in os.walk("."):
        if filename in files:
            return os.path.join(root, filename)
    parent_dir = os.path.abspath(os.path.join(".", ".."))
    for root, dirs, files in os.walk(parent_dir):
        if filename in files:
            return os.path.join(root, filename)
    return None

def standardize_columns(df):
    zip_cols = [c for c in df.columns if c.lower() in ["zip_code", "zcta", "zip", "geoid", "fips"]]
    if zip_cols:
        df = df.rename(columns={zip_cols[0]: "zip_code"})
    date_cols = [c for c in df.columns if c.lower() in ["date", "datetime", "time"]]
    if date_cols:
        df = df.rename(columns={date_cols[0]: "date"})
    case_candidates = ["cases", "new_cases", "daily_cases", "cases_new", "confirmed_cases", "covid_cases", "case_count", "cases_lag0"]
    renamed = False
    for c in case_candidates:
        if c in df.columns:
            df = df.rename(columns={c: "cases"})
            renamed = True
            break
    if not renamed:
        case_like = [c for c in df.columns if "case" in c.lower()]
        if case_like:
            df = df.rename(columns={case_like[0]: "cases"})
    pop_cols = [c for c in df.columns if c.lower() in ["pop_density_sqmi", "pop_density", "density", "pop_sqmi"]]
    if pop_cols:
        df = df.rename(columns={pop_cols[0]: "pop_density_sqmi"})
    elif "pop_density_sqmi" not in df.columns:
        df["pop_density_sqmi"] = 10000.0
    svi_cols = [c for c in df.columns if c.lower() in ["svi_score", "svi", "svi_pct"]]
    if svi_cols:
        df = df.rename(columns={svi_cols[0]: "svi_score"})
    elif "svi_score" not in df.columns:
        df["svi_score"] = 0.5
    return df

def load_or_generate_dataset():
    feat_path = find_data_file("nj_daily_features.csv.gz") or find_data_file("final_ml_feature_matrix.csv")
    commute_path = find_data_file("nj_commuter_matrix.csv")
    if feat_path:
        print(f"📖 Found feature matrix at: '{feat_path}'")
        df_feat = pd.read_csv(feat_path, compression="gzip" if feat_path.endswith(".gz") else None)
        df_feat = standardize_columns(df_feat)
        df_feat["date"] = pd.to_datetime(df_feat["date"])
        df_feat["zip_code"] = df_feat["zip_code"].astype(str).str.zfill(5)
        df_feat = df_feat.sort_values(["zip_code", "date"]).reset_index(drop=True)
        if "cases_lag1" not in df_feat.columns:
            df_feat["cases_lag1"] = df_feat.groupby("zip_code")["cases"].shift(1)
        if "cases_lag7" not in df_feat.columns:
            df_feat["cases_lag7"] = df_feat.groupby("zip_code")["cases"].shift(7)
        if "case_velocity" not in df_feat.columns:
            df_feat["case_velocity"] = df_feat.groupby("zip_code")["cases"].diff().fillna(0)
        if "case_acceleration" not in df_feat.columns:
            df_feat["case_acceleration"] = df_feat.groupby("zip_code")["case_velocity"].diff().fillna(0)
        if "seir_R_t" not in df_feat.columns:
            roll7 = df_feat.groupby("zip_code")["cases"].transform(lambda x: x.rolling(7, min_periods=1).mean())
            roll7_prev = df_feat.groupby("zip_code")["cases"].transform(lambda x: x.shift(7).rolling(7, min_periods=1).mean())
            df_feat["seir_R_t"] = np.where(roll7_prev > 0, roll7 / roll7_prev, 1.0)
            df_feat["seir_R_t"] = np.clip(df_feat["seir_R_t"], 0.2, 4.0)
        if "est_ed_visits_lag1" not in df_feat.columns:
            if "est_ed_visits" in df_feat.columns:
                df_feat["est_ed_visits_lag1"] = df_feat.groupby("zip_code")["est_ed_visits"].shift(1)
            else:
                df_feat["est_ed_visits_lag1"] = df_feat["cases_lag1"] * 0.12
        unique_zips = sorted(df_feat["zip_code"].unique())
        N = len(unique_zips)
        zip_map = {z: i for i, z in enumerate(unique_zips)}
        if commute_path:
            print(f"🗺️ Ingesting LODES commuter matrix from: '{commute_path}'")
            w_df = pd.read_csv(commute_path, index_col=0)
            W = w_df.values
            row_sums = W.sum(axis=1, keepdims=True)
            W = np.divide(W, row_sums, out=np.zeros_like(W), where=row_sums!=0)
        else:
            np.random.seed(101)
            raw_W = np.random.uniform(0.01, 0.15, size=(N, N))
            np.fill_diagonal(raw_W, 0.0)
            W = raw_W / raw_W.sum(axis=1, keepdims=True)
        if "spatial_lag_cases_lag1" not in df_feat.columns:
            pivot_lag1 = df_feat.pivot(index="date", columns="zip_code", values="cases_lag1").fillna(0)
            spatial_lag_matrix = pivot_lag1.values @ W.T
            spatial_lag_df = pd.DataFrame(spatial_lag_matrix, index=pivot_lag1.index, columns=pivot_lag1.columns)
            spatial_lag_long = spatial_lag_df.reset_index().melt(id_vars="date", var_name="zip_code", value_name="spatial_lag_cases_lag1")
            df_feat = df_feat.merge(spatial_lag_long, on=["date", "zip_code"], how="left")
        if "commuting_weighted_covid_pressure" not in df_feat.columns:
            df_feat["commuting_weighted_covid_pressure"] = df_feat["spatial_lag_cases_lag1"] * 1.25
        df_clean = df_feat.dropna().reset_index(drop=True)
        return df_clean, W, zip_map
    else:
        print("ℹ️ Cleaned feature file not found locally. Generating synthetic dataset...")
        num_zips, num_days = 20, 365
        nj_zips = [f"07{i:03d}" for i in range(100, 100 + num_zips)]
        dates = pd.date_range(start="2020-03-01", periods=num_days, freq="D")
        records = []
        np.random.seed(42)
        for z_idx, z in enumerate(nj_zips):
            base_pop = np.random.randint(25000, 85000)
            density = np.random.uniform(5000, 35000)
            svi = np.random.uniform(0.15, 0.85)
            t = np.arange(num_days)
            wave1 = 45 * np.exp(-((t - 60)**2) / (2 * 15**2))
            wave2 = 80 * np.exp(-((t - 220)**2) / (2 * 25**2))
            baseline = 5 + np.random.poisson(3, num_days)
            cases = np.maximum(0, (wave1 + wave2 + baseline) * (1 + 0.1 * z_idx)).astype(int)
            for d, c in zip(dates, cases):
                records.append({
                    "zip_code": z, "date": d, "cases": c, "population": base_pop,
                    "pop_density_sqmi": density, "svi_score": svi
                })
        df = pd.DataFrame(records).sort_values(["zip_code", "date"])
        N = num_zips
        zip_map = {z: i for i, z in enumerate(nj_zips)}
        np.random.seed(101)
        raw_W = np.random.uniform(0.01, 0.15, size=(N, N))
        np.fill_diagonal(raw_W, 0.0)
        W = raw_W / raw_W.sum(axis=1, keepdims=True)
        df["cases_lag1"] = df.groupby("zip_code")["cases"].shift(1)
        df["cases_lag7"] = df.groupby("zip_code")["cases"].shift(7)
        df["case_velocity"] = df.groupby("zip_code")["cases"].diff().fillna(0)
        df["case_acceleration"] = df.groupby("zip_code")["case_velocity"].diff().fillna(0)
        roll7 = df.groupby("zip_code")["cases"].transform(lambda x: x.rolling(7, min_periods=1).mean())
        roll7_prev = df.groupby("zip_code")["cases"].transform(lambda x: x.shift(7).rolling(7, min_periods=1).mean())
        df["seir_R_t"] = np.where(roll7_prev > 0, roll7 / roll7_prev, 1.0)
        df["seir_R_t"] = np.clip(df["seir_R_t"], 0.2, 4.0)
        df["est_ed_visits_lag1"] = df["cases_lag1"] * 0.12
        pivot_lag1 = df.pivot(index="date", columns="zip_code", values="cases_lag1").fillna(0)
        spatial_lag_matrix = pivot_lag1.values @ W.T
        spatial_lag_df = pd.DataFrame(spatial_lag_matrix, index=pivot_lag1.index, columns=pivot_lag1.columns)
        spatial_lag_long = spatial_lag_df.reset_index().melt(id_vars="date", var_name="zip_code", value_name="spatial_lag_cases_lag1")
        df = df.merge(spatial_lag_long, on=["date", "zip_code"], how="left")
        df["commuting_weighted_covid_pressure"] = df["spatial_lag_cases_lag1"] * 1.25
        df_clean = df.dropna().reset_index(drop=True)
        return df_clean, W, zip_map

In [ ]:
feature_df, W_matrix, zip_to_idx = load_or_generate_dataset()
feature_cols = ['cases_lag1', 'cases_lag7', 'spatial_lag_cases_lag1', 'commuting_weighted_covid_pressure', 'seir_R_t', 'case_velocity', 'case_acceleration', 'est_ed_visits_lag1', 'pop_density_sqmi', 'svi_score']
print(f"Loaded dataset: {len(feature_df)} rows across {feature_df['zip_code'].nunique()} ZIP codes.")

In [ ]:
# Fit Random Forest Baseline (Out-of-Sample Evaluation)
split_idx = int(len(feature_df) * 0.8)
train_df = feature_df.iloc[:split_idx]
test_df = feature_df.iloc[split_idx:].copy()

X_train = train_df[feature_cols].values
y_train = train_df["cases"].values
X_test = test_df[feature_cols].values
y_test = test_df["cases"].values

rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

preds = rf_model.predict(X_test)
rmse = calc_rmse(y_test, preds)
mae = calc_mae(y_test, preds)

print("=== Random Forest Baseline Model Fit (Out-of-Sample Evaluation) ===")
print(f"Out-of-Sample RMSE: {rmse:.4f} cases")
print(f"Out-of-Sample MAE:  {mae:.4f} cases")

# Feature Importances
importances = rf_model.feature_importances_
importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "RF Importance": importances
}).sort_values("RF Importance", ascending=False)

print("=== Random Forest Feature Importances (Leakage Free) ===")
print(importance_df.to_string(index=False))

# Plot Feature Importances
plt.figure(figsize=(10, 5))
plt.barh(importance_df["Feature"][::-1], importance_df["RF Importance"][::-1], color="#2b5c8f")
plt.title("Random Forest Feature Importance Rankings (Target Leakage Free)")
plt.xlabel("Importance Score")
plt.tight_layout()
os.makedirs("rf_baseline_outputs", exist_ok=True)
plt.savefig("rf_baseline_outputs/rf_feature_importances.png", dpi=150)
plt.close()

importance_df.to_csv("rf_baseline_outputs/rf_feature_importances.csv", index=False)
print("✅ Saved feature importances to rf_baseline_outputs/rf_feature_importances.csv")